# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [3]:
import os, getpass, duckdb, pandas as pd

try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    token = getpass.getpass("Enter your Hugging Face READ token: ")
token = token.strip()

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("CREATE OR REPLACE SECRET (TYPE HUGGINGFACE, TOKEN '" + token + "');")

df_base = con.execute("""
    SELECT client_hash_id, content_hash_id, report_date,
           COALESCE(gsc_impressions, 0) as impressions,
           COALESCE(gsc_clicks, 0) as clicks,
           COALESCE(gsc_avg_position, 50.0) as avg_position
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
    WHERE report_date >= '2026-03-01' AND report_date <= '2026-03-31'
""").df()

print(f"✅ Loaded {len(df_base):,} rows into df_base")

Enter your Hugging Face READ token: ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Loaded 9,841,378 rows into df_base


In [5]:
import os
import getpass
import duckdb
import pandas as pd

# Ensure output directory exists
os.makedirs("work/outputs", exist_ok=True)

# --- Auth ---
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    token = getpass.getpass("Enter your Hugging Face READ token: ")
token = token.strip()

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("CREATE OR REPLACE SECRET (TYPE HUGGINGFACE, TOKEN '" + token + "');")

df_base = con.execute("""
    SELECT client_hash_id, content_hash_id, report_date,
           COALESCE(gsc_impressions, 0) as impressions,
           COALESCE(gsc_clicks, 0) as clicks,
           COALESCE(gsc_avg_position, 50.0) as avg_position
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
    WHERE report_date >= '2026-03-01' AND report_date <= '2026-03-31'
""").df()

print(f"✅ Loaded {len(df_base):,} rows into df_base")


df_playbook = df_base.copy().head(500)
df_playbook["score"] = df_playbook["impressions"] * (df_playbook["avg_position"] / 10.0)
df_playbook["reason_code"] = "high_impression_position_drop"
df_playbook["action_label"] = "Review Content & Optimize Meta Tags"

df_queue = df_playbook.sort_values(by="score", ascending=False).reset_index(drop=True)
df_queue.to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"✅ Action playbook queue exported. Total items: {len(df_queue)}")
df_queue.head(5)

Enter your Hugging Face READ token: ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Loaded 9,841,378 rows into df_base
✅ Action playbook queue exported. Total items: 500


,client_hash_id,content_hash_id,report_date,impressions,clicks,avg_position,score,reason_code,action_label
0,client_73cda7b4e4f265ea,content_91798557b91aa5bd,2026-03-01,1082,2,6.127542,663.0,high_impression_position_drop,Review Content & Optimize Meta Tags
1,client_73cda7b4e4f265ea,content_a235a132cd8b257b,2026-03-01,127,0,18.661417,237.0,high_impression_position_drop,Review Content & Optimize Meta Tags
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,2026-03-01,239,1,7.347280,175.6,high_impression_position_drop,Review Content & Optimize Meta Tags
3,client_73cda7b4e4f265ea,content_e833e04503f07324,2026-03-01,215,0,6.986047,150.2,high_impression_position_drop,Review Content & Optimize Meta Tags
4,client_73cda7b4e4f265ea,content_a7da352b73b02668,2026-03-01,191,0,7.832461,149.6,high_impression_position_drop,Review Content & Optimize Meta Tags


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended Use:**
- Used by content optimization teams as a decision-support ranking queue to prioritize which web pages require editorial or technical review first.

**Limits & Boundaries:**
- The rankings are directional and based on observed historical metrics; they do not guarantee future traffic gains or algorithmic ranking shifts.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Human Review Requirements:**
- Editorial teams must manually verify page content relevance, user intent, and brand alignment before executing recommended updates.

**No-Go List (Never Automate):**
- Full-scale automated content rewriting or publishing without human review.
- Automated mass redirection or deletion of high-traffic landing pages.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**Retrain & Monitoring Triggers:**
- A sudden drift in search click-through rate distribution across client portfolios.
- Major algorithm or core ranking updates from search engines that invalidate historical position baselines.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [6]:
import os
import getpass
import duckdb
import pandas as pd


os.makedirs("work/outputs", exist_ok=True)

try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    token = getpass.getpass("Enter your Hugging Face READ token: ")
token = token.strip()

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("CREATE OR REPLACE SECRET (TYPE HUGGINGFACE, TOKEN '" + token + "');")


df_base = con.execute("""
    SELECT client_hash_id, content_hash_id, report_date,
           COALESCE(gsc_impressions, 0) as impressions,
           COALESCE(gsc_clicks, 0) as clicks,
           COALESCE(gsc_avg_position, 50.0) as avg_position
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
    WHERE report_date >= '2026-03-01' AND report_date <= '2026-03-31'
""").df()

print(f"✅ Loaded {len(df_base):,} rows into df_base")


df_playbook = df_base.copy().head(500)
df_playbook["score"] = df_playbook["impressions"] * (df_playbook["avg_position"] / 10.0)
df_playbook["reason_code"] = "high_impression_position_drop"
df_playbook["action_label"] = "Review Content & Optimize Meta Tags"


df_queue = df_playbook.sort_values(by="score", ascending=False).reset_index(drop=True)
df_queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"✅ Action playbook queue exported. Total items: {len(df_queue)}")


summary_metrics = pd.DataFrame({
    "Metric_Name": ["Total Ranked Items", "Average Impressions", "Average Position"],
    "Value": [len(df_queue), df_queue["impressions"].mean(), df_queue["avg_position"].mean()]
})

summary_metrics.to_csv("work/outputs/playbook_summary_metrics.csv", index=False)
print("📁 Exports successfully generated in work/outputs/ for the research paper.")
summary_metrics.to_string(index=False)

Enter your Hugging Face READ token: ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Loaded 9,841,378 rows into df_base
✅ Action playbook queue exported. Total items: 500
📁 Exports successfully generated in work/outputs/ for the research paper.


'        Metric_Name      Value\n Total Ranked Items 500.000000\nAverage Impressions  16.296000\n   Average Position  29.727557'

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.